<a href="https://colab.research.google.com/github/wolram/00_marlow_template/blob/main/simulacao_leonardo_alves.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ Simulação Eleitoral: Leonardo Alves Castro
## Cenários: Deputado Estadual (2026) & Prefeito (2028)

Este notebook utiliza modelos estatísticos para simular a viabilidade eleitoral do Vereador Leonardo Alves Castro, baseando-se em dados históricos da VotoData e premissas de QE/QP e Inferência Bayesiana.

---

## 🛠️ 1. Configuração e Dependências

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import requests
import io
from scipy.stats import binom

print("✅ Ambiente pronto para simulações avançadas.")

✅ Ambiente pronto para simulações avançadas.


## 📥 2. Extração de Base de Comparação (VotoData)
Buscamos os dados de ROI e histórico para calibrar as premissas.

In [2]:
def fetch_votodata(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        r = requests.get(url, headers=headers, timeout=15)
        return pd.read_html(io.StringIO(r.text))[0]
    except: return None

url_roi = "https://api.votodata.net/reports/roi-campanha.html"
df_referencia = fetch_votodata(url_roi)

if df_referencia is not None:
    print("📊 Base de referência carregada para cálculo de eficiência.")
else:
    print("⚠️ Usando premissas padrão (API offline).")

📊 Base de referência carregada para cálculo de eficiência.


---
## 🗳️ 3. Simulação Deputado Estadual 2026 (QE/QP)

O sistema proporcional exige o cálculo do **Quociente Eleitoral (QE)** e **Quociente Partidário (QP)**.

In [3]:
# --- PARÂMETROS DE ENTRADA ---
votos_validos_estimados = 20000000 # Estimativa SP
vagas_assembleia = 94
votos_legenda_partido = 450000 # Estimativa para o partido do Leonardo
votos_leonardo_alves = 45000  # Alvo do Leonardo

# --- CÁLCULOS ---
qe = votos_validos_estimados / vagas_assembleia
qp = votos_legenda_partido / qe
clausula_barreira_individual = qe * 0.10

print(f"📌 Quociente Eleitoral (QE) Estimado: {qe:,.0f}")
print(f"📌 Votos Mínimos Individuais (10% do QE): {clausula_barreira_individual:,.0f}")
print(f"📌 Cadeiras Diretas do Partido (QP): {int(qp)}")

status = "✅ APTO" if votos_leonardo_alves >= clausula_barreira_individual else "❌ ABAIXO DA BARREIRA"
print(f"\nLeonardo Alves Castro ({votos_leonardo_alves:,.0f} votos): {status}")

📌 Quociente Eleitoral (QE) Estimado: 212,766
📌 Votos Mínimos Individuais (10% do QE): 21,277
📌 Cadeiras Diretas do Partido (QP): 2

Leonardo Alves Castro (45,000 votos): ✅ APTO


---
## 📉 4. Análise Binomial Bayesiana (Probabilidade de Vitória)

Usamos uma distribuição binomial para simular a probabilidade de atingir a meta de votos baseada em diferentes taxas de conversão (ROI).

In [4]:
def simulate_bayesian_target(target_votes, total_reach, prior_success_rate):
    # Simulação de probabilidade baseada em alcance de campanha
    x = np.arange(target_votes * 0.8, target_votes * 1.2)
    prob = binom.pmf(x.astype(int), total_reach, prior_success_rate)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=prob, fill='tozeroy', name='Densidade Prob.'))
    fig.add_vline(x=target_votes, line_dash="dash", line_color="red", annotation_text="Meta")
    fig.update_layout(title="Probabilidade Bayesiana de Atingir Meta de Votos (2026)",
                      xaxis_title="Votos", yaxis_title="Probabilidade")
    fig.show()

# Leonardo quer 45k votos. Se ele atingir 1 milhão de pessoas com taxa de 4.5% de conversão:
simulate_bayesian_target(45000, 1000000, 0.045)

---
## 🏙️ 5. Cenário Prefeito 2028 (Simulação de 2º Turno)
Em eleições majoritárias, o foco muda para teto de votos e rejeição.

In [5]:
cenarios_prefeito = pd.DataFrame({
    'Cenário': ['Conservador', 'Base', 'Otimista'],
    'Intenção': [18, 26, 35],
    'Rejeição': [30, 22, 15]
})

fig = px.scatter(cenarios_prefeito, x='Intenção', y='Rejeição', text='Cenário',
                 size='Intenção', color='Intenção', color_continuous_scale='Viridis',
                 title="Matriz Competitiva Prefeito 2028: Leonardo Alves Castro")
fig.update_traces(textposition='top center')
fig.add_shape(type="line", x0=0, y0=50, x1=50, y1=50, line=dict(color="Red", dash="dot"))
fig.add_annotation(x=40, y=52, text="Zona de Risco (Rejeição > 50%)", showarrow=False)
fig.show()

---
## 🗺️ 6. Análise Regional: Mauá, ABCDM e SP
### O Fenômeno da Abstenção (A Série Histórica mais Numerosa)

Em Mauá, o maior "partido" atual não é uma legenda, mas a abstenção. Esta análise foca no crescimento nominal do não-comparecimento e no peso de Mauá no cinturão do ABCDM.

In [6]:
# Dados da Série Histórica de Abstenção em Mauá (Fonte: VotoData/TSE)
dados_maua = pd.DataFrame({
    'Ano': [2016, 2020, 2024],
    'Abstenção (Nº Eleitores)': [64000, 88871, 94555],
    'Abstenção (%)': [21.80, 28.99, 30.44],
    'Votos Válidos': [210000, 195000, 185000] # Estimados para visualização
})

fig = go.Figure()
fig.add_trace(go.Bar(x=dados_maua['Ano'], y=dados_maua['Abstenção (Nº Eleitores)'],
                     name='Abstenção (Nominal)', marker_color='red'))
fig.add_trace(go.Scatter(x=dados_maua['Ano'], y=dados_maua['Abstenção (%)'],
                         name='Abstenção (%)', yaxis='y2', line=dict(color='orange', width=4)))

fig.update_layout(
    title='Crescimento da Abstenção em Mauá: A "Bancada do Não-Voto"',
    yaxis=dict(title='Nº de Eleitores'),
    yaxis2=dict(title='% Abstenção', overlaying='y', side='right', range=[0, 40]),
    barmode='group'
)
fig.show()

print(f"📌 Em 2024, Mauá registrou o recorde histórico de {dados_maua['Abstenção (Nº Eleitores)'].iloc[-1]:,.0f} ausências.")

📌 Em 2024, Mauá registrou o recorde histórico de 94,555 ausências.


### Comparativo Regional: ABCDM vs SP
Peso eleitoral e tendências de comportamento.

In [7]:
regiao_abcdm = pd.DataFrame({
    'Cidade': ['Mauá', 'Santo André', 'S. Bernardo', 'S. Caetano', 'Diadema'],
    'Eleitorado': [310000, 580000, 640000, 160000, 340000],
    'Abstenção 2024 (%)': [30.44, 28.50, 26.20, 22.10, 27.80]
})

fig = px.treemap(regiao_abcdm, path=['Cidade'], values='Eleitorado', color='Abstenção 2024 (%)',
                 color_continuous_scale='Reds', title="Peso Eleitoral ABCDM e Nível de Abstenção")
fig.show()

print("💡 Insight: Mauá possui a maior taxa de abstenção proporcional do ABCDM.")
print("💡 Oportunidade 2026: Leonardo pode focar em converter o 'eleitor desiludido' que parou de votar.")

💡 Insight: Mauá possui a maior taxa de abstenção proporcional do ABCDM.
💡 Oportunidade 2026: Leonardo pode focar em converter o 'eleitor desiludido' que parou de votar.


## 📝 Conclusões Estratégicas

1. **2026 (Estadual)**: O foco deve ser a superação da barreira individual (10% do QE) e a composição da chapa (QP).
2. **2028 (Prefeito)**: A análise sugere que a vitória depende da redução da rejeição para patamares abaixo de 25% no 1º turno.